# Phase 6 — ETL Pipeline (Data Cleaning & Validation)

**Project:** Banking Transaction Monitoring & Fraud Analytics Platform
**Database:** `federal_bank`

**ETL = Extract, Transform, Load.** We **extract** raw incoming transactions from a
*staging* table, **transform** them by cleaning and validating, and **load** only the
clean rows into the strict `transactions` table.

**Why a staging table?** Our `transactions` table is strict (NOT NULL, CHECK, ENUM,
foreign keys), so it rejects bad data outright. Real systems therefore land raw data
in a permissive *staging* area first, validate it there, and load only clean rows into
production. This notebook demonstrates that by deliberately injecting dirty rows and
showing the pipeline catch and report them.

**The golden rule for interviews:** never run analytics on data you haven't validated —
*garbage in, garbage out.*

## 1. Configuration

In [ ]:
DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": "YOUR_MYSQL_PASSWORD",   # <-- your MySQL password
    "database": "federal_bank",
}
REJECTED_CSV = "datasets/etl_rejected_rows.csv"   # audit log of rejected rows

## 2. Imports

In [ ]:
import pandas as pd
from datetime import datetime, timedelta
import mysql.connector
import os

os.makedirs("datasets", exist_ok=True)

## 3. Connect to MySQL

In [ ]:
conn = mysql.connector.connect(**DB_CONFIG)
cursor = conn.cursor()
print("Connected to database:", DB_CONFIG["database"])

## 4. Gather reference facts
We need the set of **valid account ids** (to detect orphan references) and the
**date range** of real transactions (to detect out-of-range timestamps).

In [ ]:
cursor.execute("SELECT account_id FROM accounts")
valid_account_ids = {row[0] for row in cursor.fetchall()}

cursor.execute("SELECT MIN(transaction_timestamp), MAX(transaction_timestamp) FROM transactions")
min_ts, max_ts = cursor.fetchone()
DATE_MIN = min_ts - timedelta(days=1)
DATE_MAX = max_ts + timedelta(days=1)
print(f"{len(valid_account_ids)} valid accounts. Valid date range: {DATE_MIN} to {DATE_MAX}")

## 5. Create the staging table
A permissive copy of the transaction fields — every column is text and nullable,
so it can hold *raw, possibly dirty* data (this is what real "landing" tables do).

In [ ]:
cursor.execute("DROP TABLE IF EXISTS transactions_staging")
cursor.execute("""
    CREATE TABLE transactions_staging (
        staging_id            INT AUTO_INCREMENT PRIMARY KEY,
        account_id            VARCHAR(20),
        card_id               VARCHAR(20),
        transaction_type      VARCHAR(50),
        channel               VARCHAR(50),
        amount                VARCHAR(30),
        currency              VARCHAR(10),
        status                VARCHAR(30),
        transaction_city      VARCHAR(50),
        counterparty_account  VARCHAR(30),
        description           VARCHAR(255),
        transaction_timestamp VARCHAR(40)
    ) ENGINE=InnoDB
""")
conn.commit()
print("Staging table ready.")

## 6. Inject raw data: some clean, some deliberately dirty
Each dirty row isolates one common data-quality problem so the report is easy to read.

In [ ]:
ids = sorted(valid_account_ids)[:5]          # a few real accounts
orphan_id = max(valid_account_ids) + 100000  # an account id that does NOT exist

def base(aid, **overrides):
    # A valid row; override one field to create a specific problem.
    row = dict(account_id=str(aid), card_id="", transaction_type="Payment",
               channel="POS", amount="1200", currency="INR", status="Success",
               transaction_city="Pune", counterparty_account="",
               description="incoming txn", transaction_timestamp="2026-01-03 14:00:00")
    row.update(overrides)
    return row

raw_batch = []
# --- Clean rows (should PASS) ---
for i, aid in enumerate(ids):
    raw_batch.append(base(aid, channel="UPI", amount=str(500 + i * 10),
                          transaction_timestamp="2026-01-02 09:%02d:00" % i))
# A clean row that just needs normalizing (lowercase channel, padded/space status) — should PASS after transform
raw_batch.append(base(ids[0], transaction_type="payment", channel="upi", status=" success ",
                      transaction_city="  Delhi  ", amount="750",
                      transaction_timestamp="2026-01-02 09:30:00"))

# --- Dirty rows (should be REJECTED) ---
raw_batch.append(base(ids[1], account_id=None))                 # missing account_id
raw_batch.append(base(ids[1], amount="-500"))                   # negative amount
raw_batch.append(base(ids[1], amount="0"))                      # zero amount
raw_batch.append(base(ids[1], amount="abc"))                    # non-numeric amount
raw_batch.append(base(ids[1], status="Sucess"))                # invalid status (typo)
raw_batch.append(base(ids[1], channel="Cheque"))              # invalid channel
raw_batch.append(base(orphan_id))                              # orphan account id
raw_batch.append(base(ids[1], transaction_timestamp="not-a-date"))       # invalid timestamp
raw_batch.append(base(ids[1], transaction_timestamp="2030-01-01 12:00:00"))  # out-of-range date
raw_batch.append(dict(raw_batch[0]))                          # exact duplicate of first clean row

# Insert everything into staging.
insert_cols = ["account_id","card_id","transaction_type","channel","amount","currency",
               "status","transaction_city","counterparty_account","description","transaction_timestamp"]
placeholders = ", ".join(["%s"] * len(insert_cols))
cursor.executemany(
    f"INSERT INTO transactions_staging ({', '.join(insert_cols)}) VALUES ({placeholders})",
    [tuple(r.get(c) for c in insert_cols) for r in raw_batch],
)
conn.commit()
print(f"Injected {len(raw_batch)} raw rows into staging "
      f"({len(ids)+1} clean, {len(raw_batch)-len(ids)-1} dirty).")

## 7. Extract raw rows from staging

In [ ]:
dict_cursor = conn.cursor(dictionary=True)
dict_cursor.execute(f"SELECT {', '.join(insert_cols)} FROM transactions_staging")
raw_rows = dict_cursor.fetchall()
dict_cursor.close()
print(f"Extracted {len(raw_rows)} rows from staging.")

## 8. Transform & validate
Trims whitespace, standardizes case, coerces types, and checks every rule.
Rows with any problem are rejected (with reasons); clean rows are normalized.

In [ ]:
ALLOWED_TYPES    = {"Deposit", "Withdrawal", "Transfer", "Payment", "Reversal"}
ALLOWED_CHANNELS = {"ATM", "POS", "Internet", "Mobile", "Branch", "UPI"}
ALLOWED_STATUS   = {"Success", "Failed", "Reversed"}

# Maps a lower-cased value back to its canonical spelling, e.g. "atm" -> "ATM".
CANON_TYPE    = {v.lower(): v for v in ALLOWED_TYPES}
CANON_CHANNEL = {v.lower(): v for v in ALLOWED_CHANNELS}
CANON_STATUS  = {v.lower(): v for v in ALLOWED_STATUS}


def _clean_text(x):
    # None -> "", and trim surrounding whitespace.
    return (x or "").strip() if not isinstance(x, str) else x.strip()


def clean_and_validate(rows, valid_ids, date_min, date_max):
    clean, rejected, seen = [], [], set()
    for row in rows:
        problems = []

        acct    = _clean_text(row.get("account_id"))
        card    = _clean_text(row.get("card_id"))
        ttype   = _clean_text(row.get("transaction_type"))
        channel = _clean_text(row.get("channel"))
        amt_raw = _clean_text(row.get("amount"))
        currency = _clean_text(row.get("currency")) or "INR"
        status  = _clean_text(row.get("status"))
        city    = _clean_text(row.get("transaction_city")) or None
        cp      = _clean_text(row.get("counterparty_account")) or None
        desc    = _clean_text(row.get("description")) or None
        ts_raw  = _clean_text(row.get("transaction_timestamp"))

        # Standardize case to canonical spelling where possible.
        ttype   = CANON_TYPE.get(ttype.lower(), ttype)
        channel = CANON_CHANNEL.get(channel.lower(), channel)
        status  = CANON_STATUS.get(status.lower(), status)

        # account_id: required, numeric, and must exist
        if not acct:
            problems.append("missing account_id")
        elif not acct.isdigit():
            problems.append("non-numeric account_id")
        elif int(acct) not in valid_ids:
            problems.append("orphan account_id")

        # amount: numeric and positive
        amt = None
        try:
            amt = float(amt_raw)
            if amt <= 0:
                problems.append("amount not positive")
        except ValueError:
            problems.append("non-numeric amount")

        # controlled vocabularies
        if ttype not in ALLOWED_TYPES:
            problems.append("invalid transaction_type")
        if channel not in ALLOWED_CHANNELS:
            problems.append("invalid channel")
        if status not in ALLOWED_STATUS:
            problems.append("invalid status")

        # timestamp: parseable and in range
        try:
            ts = datetime.strptime(ts_raw, "%Y-%m-%d %H:%M:%S")
            if not (date_min <= ts <= date_max):
                problems.append("timestamp out of range")
        except ValueError:
            problems.append("invalid timestamp")

        # card_id: optional, but numeric if present
        card_val = None
        if card:
            try:
                card_val = int(float(card))
            except ValueError:
                problems.append("non-numeric card_id")

        if problems:
            rejected.append({**row, "reasons": "; ".join(problems)})
            continue

        # de-duplicate within the batch
        key = (acct, ts_raw, round(amt, 2), channel)
        if key in seen:
            rejected.append({**row, "reasons": "duplicate row"})
            continue
        seen.add(key)

        clean.append((
            int(acct), card_val, ttype, channel, round(amt, 2), currency,
            status, city, cp, desc, ts_raw,
        ))
    return clean, rejected

## 9. Run the pipeline

In [ ]:
clean_rows, rejected_rows = clean_and_validate(raw_rows, valid_account_ids, DATE_MIN, DATE_MAX)
print(f"Extracted : {len(raw_rows)}")
print(f"Clean     : {len(clean_rows)}")
print(f"Rejected  : {len(rejected_rows)}")

## 10. Data-quality report
A breakdown of *why* rows were rejected — the kind of report a data analyst hands to
stakeholders to prove data quality.

In [ ]:
if rejected_rows:
    rej_df = pd.DataFrame(rejected_rows)
    print("Rejections by reason:\n")
    print(rej_df["reasons"].value_counts().to_string())
    rej_df.to_csv(REJECTED_CSV, index=False)
    print(f"\nSaved rejected rows to {REJECTED_CSV}")
else:
    print("No rejected rows.")

## 11. Load the clean rows into the transactions table
Only validated rows reach production — this is the "Load" in ETL.

In [ ]:
LOAD_SQL = (
    "INSERT INTO transactions "
    "(account_id, card_id, transaction_type, channel, amount, currency, status, "
    " transaction_city, counterparty_account, description, transaction_timestamp) "
    "VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)"
)
if clean_rows:
    cursor.executemany(LOAD_SQL, clean_rows)
    conn.commit()
    print(f"Loaded {len(clean_rows)} clean rows into the transactions table.")
else:
    print("Nothing to load.")

## 12. Verify and close

In [ ]:
cursor.execute("SELECT COUNT(*) FROM transactions")
print("Total rows now in transactions:", cursor.fetchone()[0])
cursor.close()
conn.close()
print("ETL complete. Clean data is ready for fraud detection (Phase 7).")